In [28]:
import pandas as pd

# Read in dataset from articles dataset
articles_df = pd.read_csv("../data/articles.csv")
articles_df

,match_id,source,url,text
0,43,espn,https://www.espn.co.uk/football/report/_/gameI...,Man United v Brighton\nMan United beat Brighto...
1,43,guardian,https://www.theguardian.com/football/2025/oct/...,A Ruben Amorim pirouette and revolving fist-pu...
2,43,skysports,https://www.skysports.com/football/news/11661/...,Man Utd 4-2 Brighton: Matheus Cunha scores fir...
3,44,guardian,https://www.theguardian.com/football/2025/dec/...,From near-total control to collapse to late Br...
4,44,espn,https://www.espn.co.uk/football/match/_/gameId...,Man United v Bournemouth\nFormations & Lineups...
5,44,bbc,https://www.bbc.co.uk/sport/football/live/c4g6...,Goodnightpublished at 22:45 GMT 15 December 20...


In [ ]:
# Function to extracts the text from the relevant articles in the dataset for a match

def get_match_articles(match_id):
    return articles_df[articles_df["match_id"]==match_id]

# Test function
get_match_articles(43)

,match_id,source,url,text
0,43,espn,https://www.espn.co.uk/football/report/_/gameI...,Man United v Brighton\nMan United beat Brighto...
1,43,guardian,https://www.theguardian.com/football/2025/oct/...,A Ruben Amorim pirouette and revolving fist-pu...
2,43,skysports,https://www.skysports.com/football/news/11661/...,Man Utd 4-2 Brighton: Matheus Cunha scores fir...


In [26]:
# Funtion that joins the text from all the different articles together

def build_context(match_articles):
    texts = match_articles["text"].tolist()
    return "\n\n".join(texts)

In [29]:
# Load in the cleaned dataset as a source of truth for many of the desired feature

matches_df = pd.read_csv("../data/clean_matches.csv")
matches_df = matches_df.iloc[42:44]
matches_df

,match_id,date,ground,home team,home score,away team,away score,goalscorer(s)
42,43,2025-10-25,Old Trafford,Man United,4,Brighton,2,"['Matheus Cunha', 'Casemiro', 'Bryan Mbeumo (x..."
43,44,2025-12-15,Old Trafford,Man United,4,Bournemouth,4,"['Amad Diallo', 'Antoine Semenyo', 'Casemiro',..."


In [24]:
def get_known_facts(match_id):
    row = matches_df[matches_df["match_id"] == match_id]
    row = row.iloc[0]
 
    return {
        "home_team": row["home team"],
        "away_team": row["away team"],
        "score": f'{row["home score"]}-{row["away score"]}',
        "ground": row["ground"],
        "known_goalscorers": row["goalscorer(s)"]
    }

get_known_facts(43)

{'home_team': 'Man United',
 'away_team': 'Brighton',
 'score': '4-2',
 'ground': 'Old Trafford',
 'known_goalscorers': "['Matheus Cunha', 'Casemiro', 'Bryan Mbeumo (x2)', 'Danny Welbeck', 'Charalampos Kostoulas']"}

In [25]:
# Function that builds the query that will be fed into the LLM

# Edit OUTPUT FORMAT when a decision has been made about what summary date we want

def build_query(context):
    return f"""
    You are a strict information extraction system operating on football match reports.
 
    KNOWN FACTS (already verified from the attendee's own records — do not re-derive these,
    use them only to disambiguate which match/players the text is referring to):
    - Home team: {known_facts['home_team']}
    - Away team: {known_facts['away_team']}
    - Final score: {known_facts['score']}
    - Ground: {known_facts['ground']}
    - Known goalscorers: {', '.join(known_facts['known_goalscorers']) or 'none recorded'}
 
    TASK:
    Extract ONLY the following, which are NOT already known:
    - The minute and scoring team for each goal
    - Any red cards
    - Home and away managers
    - Man of the match
    - Any stadium detail that adds to or conflicts with the known ground
 
    RULES:
    - Use ONLY information explicitly stated in the TEXT below.
    - Do NOT guess or infer missing data. If a field is not stated, leave it as an empty
      string or empty list.
    - Do NOT invent players, minutes, or names that are not present in the text.
    - Do NOT repeat or duplicate events.
    - Every fact you return MUST include a short verbatim quote (<=25 words, copied exactly
      from the TEXT) as its "evidence". If you cannot find a supporting quote, omit the fact.
 
    TEXT:
    {context}
    """.strip()

In [30]:
# Get API key for OpenRouter from .env file

import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")

In [ ]:
# JSON format that we want output to take

RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "match_extraction",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "Home team": {"type": "string"},
                "Away team": {"type": "string"},
                "Final score": {"type": "string"},
                "Ground": {"type": "string"},
                "goals": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "minute": {"type": "string"},
                            "player": {"type": "string"},
                            "team": {"type": "string"},
                            "evidence": {"type": "string"},
                        },
                        "required": ["minute", "player", "team", "evidence"],
                        "additionalProperties": False,
                    },
                },
                "red_cards": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "minute": {"type": "string"},
                            "player": {"type": "string"},
                            "team": {"type": "string"},
                            "evidence": {"type": "string"},
                        },
                        "required": ["minute", "player", "team", "evidence"],
                        "additionalProperties": False,
                    },
                },
                "home_manager": {"type": "string"},
                "home_manager_evidence": {"type": "string"},
                "away_manager": {"type": "string"},
                "away_manager_evidence": {"type": "string"},
                "man_of_the_match": {"type": "string"},
                "man_of_the_match_evidence": {"type": "string"},
            },
            "required": [
                "Home team", "Away team", "Final score",
                "Ground", "goals", "red_cards",
                "home_manager", "home_manager_evidence",
                "away_manager", "away_manager_evidence",
                "man_of_the_match", "man_of_the_match_evidence",
            ],
            "additionalProperties": False,
        },
    },
}

In [ ]:
# Python function that interacts with the LLM

# Reasons why OpenRouter was chosen as an API
# Access to multiple models (OpenAI, Google, HuggingFace) using one API
#       Flexibility identified as important at early stage of project
#       Avoids vendor lock in
# Inexpensive

# Reasons why openai/gpt-4o-mini was chosen as AI Model.
# Relaible: Widely supported across different APIs (crucial for early stages of project when archeticeture can change)
# Excellent for following strict output formats and JSON schema adherance
# Strong for long context reasoning
# Inexpensive: Works on cheap API tiers
# Fast enough for real time use (could get away with slower times since pipeline will only run infrequently)
# Note: not open source unlike Llama 3

import requests

def call_llm(query, retries=2):
    for attempt in range(retries + 1):
        response = requests.post(                                       # Send data to server
            url="https://openrouter.ai/api/v1/chat/completions",        # URL where OpenRouter recieves and sends responses
            headers={
                "Authorization": f"Bearer {API_KEY}",                   # Communicates API Key
                "Content-Type": "application/json",                     # Telling API we sending json data
            },
            json={
                "model": "openai/gpt-4o-mini",                          # The AI model we want to use
                "temperature": 0,                                       # temperature parameter determines the randomness of the models selection. Setting to 0 makes it as deterministic as possible. Recommended for consistent data extraction
                "max_tokens": 1200,                                     # Restricts the length of the response.... REQUIRED?
                "response_format": RESPONSE_SCHEMA,                     # Structure for the output enforced by the API
                "messages": [
                    {"role": "user", "content": query}
                ],
            },
            timeout=30,                                                 # Dictates the length of time the code will wait for the API to respond...... REQUIRED? TOO SHORT?
        )
        data = response.json()
 
        if "choices" not in data:
            if attempt == retries:
                return None, f"API error: {data}"
            time.sleep(1.5 * (attempt + 1))
            continue
 
        content = data["choices"][0]["message"]["content"]              #Output given by the AI. Defaults to the first response if the AI suggests multiple models
        try:
            return json.loads(content), None
        except json.JSONDecodeError as e:
            if attempt == retries:
                return None, f"JSON parse failed after {retries + 1} attempts: {e}"
            time.sleep(1)
 
    return None, "Unknown failure"

# TO DO: ADD ERROR PROCESSING DOWNSTREAM

In [49]:
# Composes the precedding functions to generate required summary for the sample matches

import json

output = []
for match_id,group in articles_df.groupby("match_id"):
    known_facts = get_known_facts(match_id)
    match_articles = get_match_articles(match_id)
    context = build_context(match_articles)
    query = build_query(context)
    LLM_ouput = call_llm(query)[0]
    output.append(LLM_ouput)

output

[{'Home team': 'Man United',
  'Away team': 'Brighton',
  'Final score': '4-2',
  'Ground': 'Old Trafford',
  'goals': [{'minute': '24th',
    'player': 'Matheus Cunha',
    'team': 'Man United',
    'evidence': 'Cunha kept his cool after some battling play, taking a touch on the edge of the box and sending a precise right-footed effort beyond Bart Verbruggen.'},
   {'minute': '34th',
    'player': 'Casemiro',
    'team': 'Man United',
    'evidence': 'Casemiro, whose long-range strike hit Yasin Ayari and wrongfooted Verbruggen.'},
   {'minute': '61st',
    'player': 'Bryan Mbeumo',
    'team': 'Man United',
    'evidence': "Mbeumo, who squeezed a low shot through Dunk's legs and past Verbruggen at his near post in front of the Stretford End in the 61st minute."},
   {'minute': '74th',
    'player': 'Danny Welbeck',
    'team': 'Brighton',
    'evidence': 'Welbeck fired home the resulting free-kick in the 74th minute.'},
   {'minute': '90+6th',
    'player': 'Charalampos Kostoulas',
  

In [50]:
# Saves output from LLM query into a json file

import json
with open("../data/match.json","w") as f:
    json.dump(output,f,indent=4)          # indent=4 means each indentation level in the json file is presented with 4 spaces for readibility